### SQL Database Creation and Parsing

In [2]:
import sqlite3
import os

os.makedirs("data/databases", exist_ok=True)


In [3]:
conn = sqlite3.connect('data/databases/company.db')
cursor = conn.cursor()

In [5]:
cursor.execute('''CREATE TABLE IF NOT EXISTS employees
               (id INTEGER PRIMARY KEY, name TEXT, role TEXT, department TEXT, salary REAL)''')

In [6]:
cursor.execute('''CREATE TABLE IF NOT EXISTS projects
               (id INTEGER PRIMARY KEY, name TEXT, status TEXT, budget REAL, lead_id INTEGER)''')

In [7]:
# Insert sample data
employees = [
    (1, 'John Doe', 'Senior Developer', 'Engineering', 95000),
    (2, 'Jane Smith', 'Data Scientist', 'Analytics', 105000),
    (3, 'Mike Johnson', 'Product Manager', 'Product', 110000),
    (4, 'Sarah Williams', 'DevOps Engineer', 'Engineering', 98000)
]

projects = [
    (1, 'RAG Implementation', 'Active', 150000, 1),
    (2, 'Data Pipeline', 'Completed', 80000, 2),
    (3, 'Customer Portal', 'Planning', 200000, 3),
    (4, 'ML Platform', 'Active', 250000, 2)
]

In [8]:
cursor.executemany('INSERT OR REPLACE INTO employees VALUES (?,?,?,?,?)', employees)
cursor.executemany('INSERT OR REPLACE INTO projects VALUES (?,?,?,?,?)', projects)

In [9]:
cursor.execute('SELECT * FROM employees')

In [10]:
conn.commit()
conn.close()

In [13]:
from langchain_community.utilities import SQLDatabase
from langchain_community.document_loaders import SQLDatabaseLoader

db = SQLDatabase.from_uri('sqlite:///data/databases/company.db')

print(f"Tables: {db.get_usable_table_names()}")
print(f"\nTable DDL:")
# print(db.get_table_info())
loader = SQLDatabaseLoader(query="SELECT * FROM employees", db=db).load()
print(loader)


Tables: ['employees', 'projects']

Table DDL:
[Document(metadata={}, page_content='id: 1\nname: John Doe\nrole: Senior Developer\ndepartment: Engineering\nsalary: 95000.0'), Document(metadata={}, page_content='id: 2\nname: Jane Smith\nrole: Data Scientist\ndepartment: Analytics\nsalary: 105000.0'), Document(metadata={}, page_content='id: 3\nname: Mike Johnson\nrole: Product Manager\ndepartment: Product\nsalary: 110000.0'), Document(metadata={}, page_content='id: 4\nname: Sarah Williams\nrole: DevOps Engineer\ndepartment: Engineering\nsalary: 98000.0')]


In [25]:
from typing import List
from langchain_core.documents import Document
def sql_to_documents(path: str) -> List[Document]:
    conn = sqlite3.connect(path)
    cursor = conn.cursor()
    documents = []
    cursor.execute('SELECT name FROM sqlite_master WHERE type="table";')
    tables = cursor.fetchall()
    for table in tables:
        table_name= table[0]
        cursor.execute(f"PRAGMA table_info({table_name})")
        columns = cursor.fetchall()
        col_name = [col[1] for col in columns]
        cursor.execute(f'SELECT * FROM {table_name}')
        rows = cursor.fetchall()
        table_content = f"Table: {table_name}\n"
        table_content += f"Columns: {", ".join(col_name)}\n"
        table_content += f"Total Records: {len(rows)}\n\n"
        table_content += "Sample Records:\n"
        for row in rows[:5]:
            record = dict(zip(col_name, row))
            table_content += f"{record}\n"
        doc = Document(
            page_content= table_content,
            metadata={
                "source": path,
                "table_name": table_name,
                "num_records": len(rows),
                "data_type": 'sql_table'
            }
        ) 
        documents.append(doc)  
        cursor.execute("""
            SELECT e.name, e.role, p.name as project_name, p.status 
            FROM employees e 
            JOIN projects p on e.id = p.lead_id 
        """) 
        relationship = cursor.fetchall()
        rel_content = "Employee-Project Relationships:\n\n"
        for rel in relationship:
            rel_content += f"{rel[0]} {rel[1]} leads -> {rel[2]} Status: {rel[3]}"
        rel_doc = Document(
            page_content= rel_content,
            metadata = {
                'source': path,
            'data_type': 'sql_relationships',
            'query': 'employee_project_join'
            }
        )    
        documents.append(rel_doc)
    return documents  
sql_to_documents('data/databases/company.db')    

[Document(metadata={'source': 'data/databases/company.db', 'table_name': 'employees', 'num_records': 4, 'data_type': 'sql_table'}, page_content="Table: employees\nColumns: id, name, role, department, salary\nTotal Records: 4\n\nSample Records:\n{'id': 1, 'name': 'John Doe', 'role': 'Senior Developer', 'department': 'Engineering', 'salary': 95000.0}\n{'id': 2, 'name': 'Jane Smith', 'role': 'Data Scientist', 'department': 'Analytics', 'salary': 105000.0}\n{'id': 3, 'name': 'Mike Johnson', 'role': 'Product Manager', 'department': 'Product', 'salary': 110000.0}\n{'id': 4, 'name': 'Sarah Williams', 'role': 'DevOps Engineer', 'department': 'Engineering', 'salary': 98000.0}\n"),
 Document(metadata={'source': 'data/databases/company.db', 'data_type': 'sql_relationships', 'query': 'employee_project_join'}, page_content='Employee-Project Relationships:\n\nJohn Doe Senior Developer leads -> RAG Implementation Status: ActiveJane Smith Data Scientist leads -> Data Pipeline Status: CompletedMike Joh